## EDA: 데이터 구조 파악

In [ ]:
import csv
from collections import Counter
import json
import statistics
from collections import defaultdict, Counter

### Step 1: 클래스 분포

In [ ]:
# 1. read train csv
labels = []
for row in csv.DictReader(open("../data/train_labels.csv", encoding="utf-8")):
    labels.append(row["action"])

# 2. count labels
counter = Counter(labels)
total = len(labels)

# 3. print
for action, cnt in sorted(counter.items(), key=lambda x: -x[1]):
    print(f"{action}: {cnt}건 ({cnt/total:.2%})")

edit_file: 11171건 (15.96%)
grep_search: 9912건 (14.16%)
read_file: 9257건 (13.22%)
glob_pattern: 5284건 (7.55%)
respond_only: 5178건 (7.40%)
run_bash: 5068건 (7.24%)
apply_patch: 4823건 (6.89%)
run_tests: 4561건 (6.52%)
list_directory: 4329건 (6.18%)
ask_user: 2701건 (3.86%)
plan_task: 2679건 (3.83%)
lint_or_typecheck: 2283건 (3.26%)
write_file: 1481건 (2.12%)
web_search: 1273건 (1.82%)


### Step 2: current_prompt 길이 분포

In [5]:
# 1. read train csv
labels = {}
for row in csv.DictReader(open('../data/train_labels.csv', encoding='utf-8')):
    labels[row['id']] = row['action']

# 2. read train jsonl
samples = []
for line in open('../data/train.jsonl', encoding='utf-8'):
    if line.strip():
        samples.append(json.loads(line))

# 3. prompt lenths by action
action_lens = defaultdict(list)
for s in samples:
    action = labels[s['id']]
    length = len(s['current_prompt'])
    action_lens[action].append(length)

# 4. total statics
all_lens = [len(s['current_prompt']) for s in samples]
print(f'전체 평균: {statistics.mean(all_lens):.1f}자')
print(f'전체 최소: {min(all_lens)}자')
print(f'전체 최대: {max(all_lens)}자')
print()

# 5. print avg lenth by action
for action, lens in sorted(action_lens.items(), key=lambda x: -statistics.mean(x[1])):
    print(f'{action:<25} 평균 {statistics.mean(lens):.1f}자')


전체 평균: 61.0자
전체 최소: 2자
전체 최대: 346자

plan_task                 평균 87.6자
ask_user                  평균 85.8자
web_search                평균 85.2자
write_file                평균 72.2자
apply_patch               평균 66.2자
edit_file                 평균 65.6자
glob_pattern              평균 64.7자
grep_search               평균 64.4자
list_directory            평균 64.1자
read_file                 평균 62.9자
respond_only              평균 42.9자
run_tests                 평균 40.1자
lint_or_typecheck         평균 39.5자
run_bash                  평균 39.2자


| 그룹 | action | 평균 길이 |
|------|--------|-----------|
| 긴 그룹 | plan_task, ask_user, web_search | 85~88자 |
| 짧은 그룹 | run_bash, lint_or_typecheck, run_tests | 39~43자 |

**인사이트**
- 전체 평균 61자, 최소 2자, 최대 346자
- 짧은 그룹과 긴 그룹이 약 2배 차이로 명확히 구분됨
- → current_prompt 길이를 피처로 추가 예정

### Step 3: history 패턴 분석